# Camera Models

A camera model describes how a 3D point in the camera coordinate frame `(x, y, z)` is projected onto a 2D point `(u, v)` in the image. Different lenses (standard, wide-angle, fisheye, panoramic) obey different projection laws, so several models are used in practice.

## Projection Models

Before lens distortion is added, most models are built on one of a few ideal projection laws. Let `θ` be the angle between an incoming ray and the optical axis, and `r` the radial distance of the projected point from the principal point. The mapping `r = f(θ)` is what distinguishes the projection types:

| Projection | Radial mapping | Notes / typical use |
| --- | --- | --- |
| Perspective (rectilinear / pinhole) | `r = f · tan θ` | Standard lenses; straight lines stay straight; `tan θ → ∞` at `θ = 90°`, so it cannot cover ≥ 180° FOV |
| Stereographic | `r = 2f · tan(θ/2)` | Some wide-angle fisheyes; conformal (locally preserves angles) |
| Equidistant | `r = f · θ` | Common fisheye "ideal"; image radius is proportional to incidence angle |
| Equisolid angle | `r = 2f · sin(θ/2)` | Many physical fisheye lenses; preserves relative area |
| Orthographic | `r = f · sin θ` | Fisheye; field of view limited to 180° |

Two further approximations of the perspective model are common in structure-from-motion:

- **Weak perspective**: the per-point perspective division by `z` is replaced by division by a single average depth, so all points are scaled by one factor. Valid when the object's depth extent is small compared with its distance from the camera.
- **Orthographic (parallel) projection**: the limit of perspective projection as focal length and object distance grow without bound; depth is dropped entirely (`u = f · x`, `v = f · y`). A good approximation for telephoto lenses viewing distant, shallow scenes.

## Perspective Camera

<img src="https://latex.codecogs.com/svg.latex?%5Cbegin%7Barray%7D%7Bl%7D%20x_n%20%3D%20%5Cfrac%7Bx%7D%7Bz%7D%20%5C%5C%20y_n%20%3D%20%5Cfrac%7By%7D%7Bz%7D%20%5C%5C%20r%5E2%20%3D%20x_n%5E2%20&plus;%20y_n%5E2%20%5C%5C%20d%20%3D%201%20&plus;%20k_1%20r%5E2%20&plus;%20k_2%20r%5E4%20%5C%5C%20u%20%3D%20f%5C%20d%5C%20x_n%20%5C%5C%20v%20%3D%20f%5C%20d%5C%20y_n%20%5Cend%7Barray%7D" alt="https://latex.codecogs.com/svg.latex?\begin{array}{l}
x_n = \frac{x}{z} \\
y_n = \frac{y}{z} \\
r^2 = x_n^2 + y_n^2 \\
d = 1 + k_1 r^2 + k_2 r^4 \\
u = f\ d\ x_n \\
v = f\ d\ y_n
\end{array}
" />

Here `x_n = x/z`, `y_n = y/z` is the ideal pinhole (rectilinear, `r = f · tan θ`) projection, and `d = 1 + k1 r² + k2 r⁴` is a radial (Brown–Conrady) distortion factor applied in the normalized image plane.

## Kannala-Brandt Fisheye Camera

Fisheye lenses have a very wide field of view (often ≥ 180°), which the rectilinear model cannot represent because `r = f · tan θ` diverges as `θ → 90°`. Fisheye models are therefore written in terms of the incidence angle `θ` instead of the projected radius. The simplest baseline is the **equidistant** projection `r = f · θ`; the Kannala-Brandt model generalizes it by adding an odd polynomial in `θ`:

<img src="https://latex.codecogs.com/svg.latex?%5Cbegin%7Barray%7D%7Bl%7D%20r%5E2%20%3D%20x%5E2%20&plus;%20y%5E2%20%5C%5C%20%5Ctheta%20%3D%20%5Carctan%28r%20/%20z%29%20%5C%5C%20d%20%3D%201%20&plus;%20k_1%20%5Ctheta%5E2&plus;%20k_2%20%5Ctheta%5E4%20%5C%5C%20u%20%3D%20f%5C%20d%5C%20%5Ctheta%5C%20%5Cfrac%7Bx%7D%7Br%7D%20%5C%5C%20v%20%3D%20f%5C%20d%5C%20%5Ctheta%5C%20%5Cfrac%7By%7D%7Br%7D%20%5Cend%7Barray%7D" alt="https://latex.codecogs.com/svg.latex?\begin{array}{l}
r^2 = x^2 + y^2 \\
\theta = \arctan(r / z) \\
d = 1 +  k_1 \theta^2+  k_2 \theta^4 \\
u = f\ d\ \theta\ \frac{x}{r} \\
v = f\ d\ \theta\ \frac{y}{r}
\end{array}
" />

The full Kannala-Brandt model, used by `cv::fisheye` in OpenCV (and exposed in COLMAP as the `OPENCV_FISHEYE` model, `fx, fy, cx, cy, k1, k2, k3, k4`), uses four distortion coefficients `k1..k4`:

<img src="https://latex.codecogs.com/svg.latex?%5Cbegin%7Balign*%7D%20L%28%5Ctilde%7Bx%7D%2C%5Ctilde%7By%7D%29%20%26%3D%20%5Cfrac%7Br_d%7D%7Br%7D%20%5Cbegin%7Bbmatrix%7D%20%5Ctilde%7Bx%7D%20%5C%5C%20%5Ctilde%7By%7D%20%5Cend%7Bbmatrix%7D%20%5C%5C%20r_d%20%26%3D%20M_1%28%5Ctheta_d%29%20%5C%5C%20%5Ctheta_d%20%26%3D%20%5Ctheta%281&plus;%20k_1%5Ctheta%5E2%20&plus;%20k_2%5Ctheta%5E4%20&plus;%20k_3%5Ctheta%5E6%20&plus;%20k_4%5Ctheta%5E8%29%20%5C%5C%20%5Ctheta%20%26%3D%20%5Carctan%28r%29%20%5C%5C%20r%20%26%3D%20%5Csqrt%7B%5Ctilde%7Bx%7D%5E2%20&plus;%20%5Ctilde%7By%7D%5E2%7D%20%5Cend%7Balign*%7D" alt="https://latex.codecogs.com/svg.latex?\begin{align*} L(\tilde{x},\tilde{y}) &= \frac{r_d}{r} \begin{bmatrix} \tilde{x} \\ \tilde{y} \end{bmatrix} \\ r_d &= M_1(\theta_d) \\ \theta_d &= \theta(1+ k_1\theta^2 + k_2\theta^4 + k_3\theta^6 + k_4\theta^8) \\ \theta &= \arctan(r) \\ r &= \sqrt{\tilde{x}^2 + \tilde{y}^2} \end{align*}" />

Read the original paper [A Generic Camera Model and Calibration Method for Conventional, Wide-Angle, and Fish-Eye Lenses](https://oulu3dvision.github.io/calibgeneric/Kannala_Brandt_calibration.pdf)


Refs: [1](https://docs.nvidia.com/vpi/algo_ldc.html)


The Kannala-Brandt fisheye model, introduced by Juho Kannala and Sami S. Brandt in "A Generic Camera Model and Calibration Method for Conventional, Wide-Angle, and Fish-Eye Lenses", describes the mapping between the direction of an incoming light ray and its location in the image for lenses with a very wide field of view (typically up to and beyond 180°).

Unlike the pinhole / Brown–Conrady model, it is parameterized by the incidence angle `θ` (the angle between the ray and the optical axis) rather than by the projected radius. The distorted image radius is modeled as an odd polynomial in `θ`:

`θ_d = θ · (1 + k1·θ² + k2·θ⁴ + k3·θ⁶ + k4·θ⁸)`

The model is **purely radial** — it is symmetric about the optical axis, and the standard formulation (as implemented in OpenCV's `cv::fisheye`) does **not** include tangential (decentering) distortion terms `p1, p2`; those belong to the Brown–Conrady pinhole model, not to Kannala-Brandt. The projected point is `θ_d` scaled by the focal length along the in-plane ray direction `(x/r, y/r)`, where `r = √(x² + y²)`.

Because a single model spans everything from conventional to fisheye lenses (by turning the higher-order coefficients on or off), the same calibration procedure — for example observing a checkerboard to establish correspondences between image and world coordinates — applies across lens types. This is the main practical advantage of the model.

**Projecting a 3D point with the Kannala-Brandt model**

```python
import numpy as np

def kannala_brandt_project(point3d, K, dist_coeffs):
    """Project a 3D camera-frame point (x, y, z) to pixel coordinates using
    the Kannala-Brandt / OpenCV cv::fisheye model.

    K           : 3x3 intrinsic matrix [[fx,0,cx],[0,fy,cy],[0,0,1]]
    dist_coeffs : [k1, k2, k3, k4] radial coefficients
    """
    x, y, z = point3d
    r = np.hypot(x, y)
    theta = np.arctan2(r, z)                     # incidence angle
    k1, k2, k3, k4 = dist_coeffs
    theta_d = theta * (1 + k1*theta**2 + k2*theta**4
                         + k3*theta**6 + k4*theta**8)
    scale = theta_d / r if r > 1e-8 else 1.0     # avoid divide-by-zero on axis
    xd, yd = x * scale, y * scale                # distorted normalized coords
    u = K[0, 0] * xd + K[0, 2]
    v = K[1, 1] * yd + K[1, 2]
    return u, v
```

Inverting this mapping (pixel to ray direction) requires solving the polynomial for `θ` given `θ_d`, which OpenCV does iteratively; the result is the undistorted ray used, for example, to rectify a fisheye image.


Refs: [1](https://oulu3dvision.github.io/calibgeneric/Kannala_Brandt_calibration.pdf)


## Spherical Camera

<img src="https://latex.codecogs.com/svg.latex?%5Cbegin%7Barray%7D%7Bl%7D%20%5Cmathrm%7Blon%7D%20%3D%20%5Carctan%5Cleft%28%5Cfrac%7Bx%7D%7Bz%7D%5Cright%29%20%5C%5C%20%5Cmathrm%7Blat%7D%20%3D%20%5Carctan%5Cleft%28%5Cfrac%7B-y%7D%7B%5Csqrt%7Bx%5E2%20&plus;%20z%5E2%7D%7D%5Cright%29%20%5C%5C%20u%20%3D%20%5Cfrac%7B%5Cmathrm%7Blon%7D%7D%7B2%20%5Cpi%7D%20%5C%5C%20v%20%3D%20-%5Cfrac%7B%5Cmathrm%7Blat%7D%7D%7B2%20%5Cpi%7D%20%5Cend%7Barray%7D" alt="https://latex.codecogs.com/svg.latex?\begin{array}{l}
\mathrm{lon} = \arctan\left(\frac{x}{z}\right) \\
\mathrm{lat} = \arctan\left(\frac{-y}{\sqrt{x^2 + z^2}}\right) \\
u = \frac{\mathrm{lon}}{2 \pi} \\
v = -\frac{\mathrm{lat}}{2 \pi}
\end{array} 
" />



Refs: [1](https://opensfm.readthedocs.io/en/latest/geometry.html)
